# fetch

> Download YouTube xscripts and metadata locally via yt-dlp

In [ ]:
#| default_exp fetch

## Storage design

**Storage location:** `~/.cache/yttoc/{video_id}/` (XDG Base Directory spec — re-downloadable data belongs in cache). Overridable via `--root`.

```
~/.cache/yttoc/
  {video_id}/
    meta.json
    captions.en.srt        # manual if available, otherwise auto-generated
```

**meta.json fields:**
```json
{
  "id": "kwSVtQ7dziU",
  "title": "Skill Issue: Andrej Karpathy on Code Agents...",
  "channel": "No Priors",
  "duration": 3991,
  "upload_date": "20260320",
  "webpage_url": "https://www.youtube.com/watch?v=kwSVtQ7dziU",
  "description": "Video description with possible manual ToC...",
  "caption_type": "auto",
  "last_used_at": "2026-04-06T14:22:31+00:00"
}
```

`description` is preserved for use as background context in ToC/summary generation. Many videos include a manual ToC with timestamps in their description.

**Caption selection logic:** prefer manual English (`en`, then `en-*`) → fall back to auto English (`en`, then `en-*`) → error if neither exists.

**CLI:** `yttoc-fetch <url>` fetches one video, prints video_id to stdout. Batch via shell: `cat urls.txt | xargs -I{} yttoc-fetch {}`

## Modularize

In [ ]:
#| export
import json, os
from datetime import datetime, timezone
from pathlib import Path
import yt_dlp

_DEFAULT_ROOT = Path(os.environ.get('XDG_CACHE_HOME', Path.home() / '.cache')) / 'yttoc'

In [ ]:
#| export
def _pick_lang(tracks: dict,
               base_lang: str = 'en' # Preferred base language
              ) -> str | None: # Best-matching language key
    "Select an exact or prefix language match from yt-dlp subtitle tracks."
    tracks = tracks or {}
    if base_lang in tracks: return base_lang
    for lang in sorted(k for k in tracks if k != 'live_chat'):
        if lang.startswith(f'{base_lang}-'):
            return lang
    return None

def _find_downloaded_srt(out_dir: str | Path # Directory containing downloaded captions
                        ) -> Path | None: # Single SRT path if present
    "Locate the downloaded SRT file emitted by yt-dlp."
    matches = sorted(Path(out_dir).glob('captions*.srt'))
    if not matches: return None
    if len(matches) > 1:
        names = ', '.join(p.name for p in matches)
        raise ValueError(f'Ambiguous caption files: {names}')
    return matches[0]

def _update_last_used(meta_path: Path # Path to meta.json
                     ) -> None:
    "Update last_used_at timestamp in meta.json."
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    meta['last_used_at'] = datetime.now(timezone.utc).isoformat()
    meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')

def _build_meta(info: dict # yt-dlp info dict
               ) -> dict: # meta.json content
    "Extract fields for meta.json from yt-dlp info."
    manual_lang = _pick_lang(info.get('subtitles', {}))
    return {
        'id': info['id'],
        'title': info['title'],
        'channel': info['channel'],
        'duration': info['duration'],
        'upload_date': info['upload_date'],
        'webpage_url': info['webpage_url'],
        'description': info.get('description', ''),
        'caption_type': 'manual' if manual_lang else 'auto',
        'last_used_at': datetime.now(timezone.utc).isoformat(),
    }

def _download_srt(url: str, info: dict, out_dir: Path) -> Path:
    "Download English SRT captions to out_dir, return final path."
    manual_lang = _pick_lang(info.get('subtitles', {}))
    auto_lang = _pick_lang(info.get('automatic_captions', {}))
    selected_lang = manual_lang or auto_lang
    if selected_lang is None:
        raise ValueError(f"No English captions available for {info['id']}")

    sub_opt = 'writesubtitles' if manual_lang else 'writeautomaticsub'
    opts = {
        'skip_download': True, 'quiet': True,
        sub_opt: True,
        'subtitleslangs': [selected_lang],
        'subtitlesformat': 'srt',
        'outtmpl': str(out_dir / 'captions.%(ext)s'),
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([url])

    srt_path = out_dir / 'captions.en.srt'
    downloaded = _find_downloaded_srt(out_dir)
    if downloaded is None:
        raise FileNotFoundError(f"yt-dlp did not write an srt caption for {info['id']}")
    if downloaded != srt_path:
        downloaded.replace(srt_path)
    return srt_path

In [ ]:
# Test: helpers
assert _pick_lang({'en': []}) == 'en'
assert _pick_lang({'live_chat': [], 'en-US': []}) == 'en-US'
assert _pick_lang({'ja': []}) is None

from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    base = Path(d)
    assert _find_downloaded_srt(base) is None
    sample = base / 'captions.en-US.srt'
    sample.write_text('1', encoding='utf-8')
    assert _find_downloaded_srt(base) == sample

# Test: _build_meta
fake_info = {'id': 'X', 'title': 't', 'channel': 'c', 'duration': 1,
             'upload_date': '20260101', 'webpage_url': 'u',
             'description': 'desc', 'subtitles': {}, 'automatic_captions': {'en': []}}
m = _build_meta(fake_info)
assert m['id'] == 'X'
assert m['description'] == 'desc'
assert m['caption_type'] == 'auto'
assert 'last_used_at' in m

# Test: _update_last_used
with TemporaryDirectory() as d:
    p = Path(d) / 'meta.json'
    p.write_text('{"id":"X"}')
    _update_last_used(p)
    assert 'last_used_at' in json.loads(p.read_text())

print('ok')

In [ ]:
#| export
def get_video_info(url: str # YouTube video URL
                  ) -> dict: # yt-dlp info dict
    "Extract video metadata and caption info without downloading."
    with yt_dlp.YoutubeDL({'skip_download': True, 'quiet': True}) as ydl:
        return ydl.extract_info(url, download=False)

In [ ]:
#| network
# Test: get_video_info smoke test (requires network)
info = get_video_info('https://www.youtube.com/watch?v=kwSVtQ7dziU')
for k in ['id', 'title', 'channel', 'duration', 'upload_date', 'webpage_url']:
    assert k in info, f'missing {k}'
assert info['id'] == 'kwSVtQ7dziU'
print('ok')

In [ ]:
#| export
def fetch_video(url: str, # YouTube video URL
                info: dict, # Result of get_video_info
                root: str | Path = None, # Root download directory (default: ~/.cache/yttoc)
               ) -> Path: # Path to video directory
    "Save metadata and English srt captions for one video."
    root = Path(root) if root else _DEFAULT_ROOT
    out_dir = root / info['id']
    out_dir.mkdir(parents=True, exist_ok=True)

    srt_path = out_dir / 'captions.en.srt'
    meta_path = out_dir / 'meta.json'
    if srt_path.exists() and meta_path.exists():
        _update_last_used(meta_path)
        return out_dir

    _download_srt(url, info, out_dir)
    meta_path.write_text(json.dumps(_build_meta(info), indent=2, ensure_ascii=False), encoding='utf-8')
    return out_dir

In [ ]:
#| eval: false
# Test: fetch_video (requires network)
import shutil
test_root = '/tmp/yttoc_test'
shutil.rmtree(test_root, ignore_errors=True)

url = 'https://www.youtube.com/watch?v=kwSVtQ7dziU'
info = get_video_info(url)
out = fetch_video(url, info, root=test_root)
assert (out / 'meta.json').exists()
assert (out / 'captions.en.srt').exists()

meta = json.loads((out / 'meta.json').read_text(encoding='utf-8'))
assert meta['id'] == 'kwSVtQ7dziU'
assert 'last_used_at' in meta

shutil.rmtree(test_root)
print('ok')

## CLI

In [ ]:
#| export
from fastcore.script import call_parse

@call_parse
def yttoc_fetch(url: str, # YouTube video URL
                root: str = None, # Root download directory (default: ~/.cache/yttoc)
               ):
    "Fetch metadata and English captions for a single YouTube video."
    info = get_video_info(url)
    out = fetch_video(url, info, root)
    print(info['id'])

In [ ]:
#| export
def _fmt_duration(seconds: int # Duration in seconds
                 ) -> str: # Formatted as H:MM:SS or M:SS
    "Format seconds as human-readable duration."
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m}:{s:02d}'

In [ ]:
# Test: _fmt_duration
assert _fmt_duration(3991) == '1:06:31'
assert _fmt_duration(195) == '3:15'
assert _fmt_duration(0) == '0:00'
assert _fmt_duration(60) == '1:00'
assert _fmt_duration(3600) == '1:00:00'
print('ok')

In [ ]:
#| export
@call_parse
def yttoc_list(root: str = None, # Root directory (default: ~/.cache/yttoc)
              ):
    "List cached videos sorted by last used."
    root = Path(root) if root else _DEFAULT_ROOT
    if not root.exists(): return

    videos = []
    for d in root.iterdir():
        if not d.is_dir(): continue
        meta_path, srt_path = d / 'meta.json', d / 'captions.en.srt'
        if not (meta_path.exists() and srt_path.exists()): continue
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        videos.append(meta)

    videos.sort(key=lambda m: m.get('last_used_at', ''), reverse=True)
    for m in videos:
        ts = m.get('last_used_at', '')[:16].replace('T', ' ')
        dur = _fmt_duration(m.get('duration', 0))
        print(f"{m['id']}  {ts}  {dur:>8}  {m.get('title', '')}")

In [ ]:
# Test: yttoc_list output
import io, contextlib
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    root = Path(d)
    # Video A: older
    a = root / 'AAAA'
    a.mkdir()
    (a / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (a / 'meta.json').write_text(json.dumps({
        'id': 'AAAA', 'title': 'Old video', 'duration': 195,
        'last_used_at': '2026-01-01T00:00:00+00:00'}))
    # Video B: newer
    b = root / 'BBBB'
    b.mkdir()
    (b / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (b / 'meta.json').write_text(json.dumps({
        'id': 'BBBB', 'title': 'New video', 'duration': 3991,
        'last_used_at': '2026-04-06T15:00:00+00:00'}))
    # Video C: incomplete (no srt) — should be skipped
    c = root / 'CCCC'
    c.mkdir()
    (c / 'meta.json').write_text('{"id":"CCCC"}')

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_list(root=str(root))
    out = buf.getvalue()

    lines = [l for l in out.strip().splitlines() if l.strip()]
    assert len(lines) >= 2, f'expected at least 2 lines, got {len(lines)}'
    assert 'BBBB' in lines[0] or 'BBBB' in lines[1], 'BBBB should appear first (newer)'
    assert 'CCCC' not in out, 'incomplete video CCCC should be skipped'
    # BBBB before AAAA (newer first)
    pos_b = out.index('BBBB')
    pos_a = out.index('AAAA')
    assert pos_b < pos_a, 'BBBB (newer) should appear before AAAA (older)'
print('ok')

In [ ]:
# Test: fetch_video returns path named by video_id, writes meta.json with last_used_at
from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    fake_info = {'id': 'TEST123', 'title': 't', 'channel': 'c',
                 'duration': 60, 'upload_date': '20260101',
                 'webpage_url': 'https://example.com', 'description': '',
                 'subtitles': {}, 'automatic_captions': {'en': [{'ext': 'srt'}]}}
    out_dir = Path(d) / 'TEST123'
    out_dir.mkdir()
    (out_dir / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\ntest\n')
    (out_dir / 'meta.json').write_text('{"id":"TEST123"}')
    result = fetch_video('https://example.com', fake_info, root=d)
    assert result.name == 'TEST123'
    meta = json.loads((result / 'meta.json').read_text())
    assert 'last_used_at' in meta
print('ok')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()